In [1]:
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa


def vtlp(audio, sr, alpha=0.9, n_fft=2048, hop_length=512):
    """
    Apply a simple vocal tract length perturbation (VTLP)
    by warping the STFT frequency axis.

    Parameters
    ----------
    audio : np.ndarray
        Mono waveform.

    sr : int
        Sampling rate.

    alpha : float
        VTLP warping factor.
        Examples: 0.9, 1.1

    n_fft : int
        FFT size.

    hop_length : int
        STFT hop size.

    Returns
    -------
    np.ndarray
        VTLP-perturbed waveform.
    """

    # STFT
    stft = librosa.stft(
        audio,
        n_fft=n_fft,
        hop_length=hop_length
    )

    magnitude = np.abs(stft)
    phase = np.angle(stft)

    n_freq_bins = magnitude.shape[0]

    # Original frequency-bin positions
    original_bins = np.arange(n_freq_bins)

    # Warp frequency positions
    warped_bins = original_bins / alpha

    # Keep positions inside valid range
    warped_bins = np.clip(
        warped_bins,
        0,
        n_freq_bins - 1
    )

    warped_magnitude = np.empty_like(magnitude)

    # Interpolate each frame independently
    for frame in range(magnitude.shape[1]):

        warped_magnitude[:, frame] = np.interp(
            original_bins,
            warped_bins,
            magnitude[:, frame]
        )

    # Recombine warped magnitude with original phase
    warped_stft = (
        warped_magnitude
        * np.exp(1j * phase)
    )

    # Reconstruct waveform
    perturbed_audio = librosa.istft(
        warped_stft,
        hop_length=hop_length,
        length=len(audio)
    )

    return perturbed_audio

In [4]:
input_audio = Path(
    r"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\Real\Patient_48\001_Patient_48_10302014190442_recording.wav"
)

output_folder = Path(
    r"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\VTLP\0.9\Patient_48"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

output_audio = (
    output_folder
    / input_audio.name
)

In [5]:
audio, sr = sf.read(input_audio)

print("Sampling rate:", sr)
print("Shape:", audio.shape)
print("Duration:", len(audio) / sr)

Sampling rate: 8000
Shape: (51080,)
Duration: 6.385


In [6]:
if audio.ndim == 2:
    audio = np.mean(audio, axis=1)

In [7]:
alpha = 0.9

perturbed_audio = vtlp(
    audio,
    sr,
    alpha=alpha
)

H:\conda\envs\speech-slm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
sf.write(
    output_audio,
    perturbed_audio,
    sr
)

print("Saved:")
print(output_audio)

Saved:
H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\VTLP\0.9\Patient_48\001_Patient_48_10302014190442_recording.wav


In [ ]:
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa


# ============================================================
# SETTINGS
# ============================================================

input_root = Path(
    r"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\Real"
)

output_root = Path(
    r"H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\VTLP"
)

alpha = 0.9

# Create folder for this VTLP factor
factor_folder = output_root / str(alpha)
factor_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# VTLP FUNCTION
# ============================================================

def vtlp(audio, sr, alpha=0.9, n_fft=2048, hop_length=512):
    """
    Apply VTLP by warping the STFT frequency axis.

    audio:
        Mono waveform.

    sr:
        Sampling rate.

    alpha:
        VTLP warping factor.

    Returns:
        VTLP-perturbed waveform with approximately
        the same duration as the original.
    """

    stft = librosa.stft(
        audio,
        n_fft=n_fft,
        hop_length=hop_length
    )

    magnitude = np.abs(stft)
    phase = np.angle(stft)

    n_freq_bins = magnitude.shape[0]

    original_bins = np.arange(n_freq_bins)

    # Frequency-axis warping
    warped_bins = original_bins / alpha

    warped_bins = np.clip(
        warped_bins,
        0,
        n_freq_bins - 1
    )

    warped_magnitude = np.empty_like(magnitude)

    # Warp every STFT frame
    for frame in range(magnitude.shape[1]):

        warped_magnitude[:, frame] = np.interp(
            original_bins,
            warped_bins,
            magnitude[:, frame]
        )

    # Combine warped magnitude with original phase
    warped_stft = (
        warped_magnitude
        * np.exp(1j * phase)
    )

    # Reconstruct waveform
    perturbed_audio = librosa.istft(
        warped_stft,
        hop_length=hop_length,
        length=len(audio)
    )

    return perturbed_audio


# ============================================================
# FIND PATIENT FOLDERS
# ============================================================

patient_folders = sorted(
    folder
    for folder in input_root.iterdir()
    if folder.is_dir()
)

print(f"Found {len(patient_folders)} patient folders.")
print(f"VTLP alpha: {alpha}")
print(f"Output: {factor_folder}\n")


# ============================================================
# PROCESS ALL PATIENTS
# ============================================================

total_processed = 0
failed_files = []

for patient_index, patient_folder in enumerate(
    patient_folders,
    start=1
):

    wav_files = sorted(
        patient_folder.glob("*.wav")
    )

    print(
        f"[{patient_index}/{len(patient_folders)}] "
        f"{patient_folder.name} "
        f"({len(wav_files)} recordings)"
    )

    # Create matching patient folder
    output_patient_folder = (
        factor_folder / patient_folder.name
    )

    output_patient_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Process recordings
    # --------------------------------------------------------

    for wav_index, input_audio in enumerate(
        wav_files,
        start=1
    ):

        try:

            # Load audio
            audio, sr = sf.read(input_audio)

            # Convert stereo -> mono if necessary
            if audio.ndim == 2:
                audio = np.mean(
                    audio,
                    axis=1
                )

            # Apply VTLP
            perturbed_audio = vtlp(
                audio,
                sr,
                alpha=alpha
            )

            # Keep exact original filename
            output_audio = (
                output_patient_folder
                / input_audio.name
            )

            # Save
            sf.write(
                output_audio,
                perturbed_audio,
                sr
            )

            total_processed += 1

        except Exception as error:

            failed_files.append(
                (
                    input_audio,
                    str(error)
                )
            )

            print(
                f"    ERROR: "
                f"{input_audio.name}"
            )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINISHED")
print("=" * 60)

print(
    f"Patients: "
    f"{len(patient_folders)}"
)

print(
    f"Successfully processed: "
    f"{total_processed}"
)

print(
    f"Failed: "
    f"{len(failed_files)}"
)

if failed_files:

    print("\nFailed files:")

    for file_path, error in failed_files:

        print(f"\n{file_path}")
        print(f"    {error}")

Found 340 patient folders.
VTLP alpha: 0.9
Output: H:\Projects\speech_synthesis\speech_synthesis\samples\ALS\VTLP\0.9

[1/340] Patient_1000 (5 recordings)
[2/340] Patient_1005 (6 recordings)
[3/340] Patient_1008 (4 recordings)
[4/340] Patient_1010 (5 recordings)
[5/340] Patient_1025 (5 recordings)
[6/340] Patient_1032 (9 recordings)
[7/340] Patient_1039 (5 recordings)
[8/340] Patient_1050 (5 recordings)
[9/340] Patient_1059 (5 recordings)
[10/340] Patient_1063 (5 recordings)
[11/340] Patient_1065 (8 recordings)
[12/340] Patient_1068 (9 recordings)
[13/340] Patient_1070 (7 recordings)
[14/340] Patient_1074 (7 recordings)
[15/340] Patient_1077 (5 recordings)
[16/340] Patient_1082 (5 recordings)
[17/340] Patient_1087 (6 recordings)
[18/340] Patient_1090 (6 recordings)
[19/340] Patient_1098 (14 recordings)
[20/340] Patient_1100 (5 recordings)
[21/340] Patient_1107 (13 recordings)
[22/340] Patient_1110 (6 recordings)
[23/340] Patient_1115 (11 recordings)
[24/340] Patient_1122 (6 recordings)